In [79]:
import pandas as pd

In [55]:
df = pd.read_csv(
    "../data/raw/DataCoSupplyChainDataset.csv",
    encoding="latin1"
)

print(df.shape)

df.info()

(180519, 53)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Email                 180519 non-null  object 
 12  Customer Fname   

In [56]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "null_pct": round(df.isnull().mean()*100,2)
})

summary.sort_values(
    "null_pct",
    ascending=False
)


,dtype,null_pct
Product Description,float64,100.00
Order Zipcode,float64,86.24
Days for shipment (scheduled),int64,0.00
Days for shipping (real),int64,0.00
Type,object,0.00
Delivery Status,object,0.00
Late_delivery_risk,int64,0.00
Category Id,int64,0.00
Category Name,object,0.00
Customer City,object,0.00


# Product Master Design

## Objective

Create a product master table to support demand monitoring, inventory planning, capacity allocation, and risk analysis.

## Source

DataCo Supply Chain Dataset

## Design Logic

The original dataset contains transaction-level order records.

To support control tower reporting, a dedicated product master table was created by extracting unique product attributes.

## Result

- 118 Unique Products
- 50 Product Categories
- 11 Departments

## Key Attributes

- Product Card Id
- Product Name
- Category Name
- Department Name

In [57]:
dim_product = (
    df[
        [
            "Product Card Id",
            "Product Name",
            "Category Name",
            "Department Name"
        ]
    ]
    .drop_duplicates()
)
print(dim_product.shape)

print(dim_product["Product Card Id"].nunique())

print(dim_product["Category Name"].nunique())

(118, 4)
118
50


In [58]:
dim_product["Department Name"].value_counts()

Department Name
Outdoors              53
Footwear              15
Fitness               15
Fan Shop               9
Apparel                8
Golf                   8
Discs Shop             4
Technology             3
Book Shop              1
Pet Shop               1
Health and Beauty      1
Name: count, dtype: int64

In [59]:
dept_demand = (
    df.groupby("Department Name")
      ["Order Item Quantity"]
      .sum()
      .sort_values(ascending=False)
)

dept_demand

Department Name
Fan Shop              106165
Golf                   99297
Apparel                98181
Footwear               43400
Outdoors               26059
Fitness                 6227
Discs Shop              2026
Technology              1465
Pet Shop                 492
Book Shop                405
Health and Beauty        362
Name: Order Item Quantity, dtype: int64

# Plant Master Design

## Objective

The original DataCo dataset does not contain manufacturing plant information.

To support capacity planning and bottleneck monitoring, a virtual manufacturing network was introduced.

## Plant Structure

### P1 - Consumer_Fab

Departments:
- Apparel
- Footwear

Business Characteristics:
- Consumer-oriented products
- Stable demand patterns
- High order volume

### P2 - Sports_Fab

Departments:
- Golf
- Outdoors
- Fitness

Business Characteristics:
- Equipment-oriented products
- Capacity-intensive operations
- Seasonal demand fluctuations

### P3 - Specialty_Fab

Departments:
- Fan Shop
- Technology
- Discs Shop
- Pet Shop
- Book Shop
- Health and Beauty

Business Characteristics:
- Specialty products
- Higher demand variability
- Diverse product portfolio

## Design Rationale

Departments were grouped based on product family similarity and demand profile.

The objective is to create a realistic manufacturing network capable of supporting capacity planning, bottleneck identification, and supply chain risk monitoring.

In [60]:
dim_plant = pd.DataFrame({
    "Plant_ID": ["P1", "P2", "P3"],
    "Plant_Name": [
        "Consumer_Fab",
        "Sports_Fab",
        "Specialty_Fab"
    ],
    "Focus_Department": [
        "Apparel,Footwear",
        "Golf,Outdoors,Fitness",
        "Fan Shop,Technology,Discs Shop,Pet Shop,Book Shop,Health and Beauty"
    ],
    "Primary_Risk": [
        "Demand Volatility",
        "Capacity Bottleneck",
        "Demand Spike"
    ]
})

dim_plant

,Plant_ID,Plant_Name,Focus_Department,Primary_Risk
0,P1,Consumer_Fab,"Apparel,Footwear",Demand Volatility
1,P2,Sports_Fab,"Golf,Outdoors,Fitness",Capacity Bottleneck
2,P3,Specialty_Fab,"Fan Shop,Technology,Discs Shop,Pet Shop,Book S...",Demand Spike


In [61]:
dim_product["Department Name"] = (
    dim_product["Department Name"]
    .str.strip()
)

In [ ]:
'''
dim_product.to_csv(
    "../data/master/dim_product.csv",
    index=False
)
'''

In [ ]:
'''
dim_plant.to_csv(
    "../data/master/dim_plant.csv",
    index=False
)
'''

In [84]:
pd.read_csv(
    "../data/master/dim_product.csv"
)

,Product Card Id,Product Name,Category Name,Department Name
0,1360,Smart watch,Sporting Goods,Fitness
1,365,Perfect Fitness Perfect Rip Deck,Cleats,Apparel
2,627,Under Armour Girls' Toddler Spine Surge Runni,Shop By Sport,Golf
3,502,Nike Men's Dri-FIT Victory Golf Polo,Women's Apparel,Golf
4,278,Under Armour Men's Compression EV SL Slide,Electronics,Footwear
...,...,...,...,...
113,646,Merrell Women's Grassbow Sport Hiking Shoe,Men's Golf Clubs,Outdoors
114,1361,Toys,Toys,Fan Shop
115,1073,Pelican Sunstream 100 Kayak,Water Sports,Fan Shop
116,1059,Pelican Maverick 100X Kayak,Water Sports,Fan Shop


In [ ]:
pd.read_csv(
    "../data/master/dim_plant.csv"
)

In [64]:
department_to_plant = {
    "Apparel": "P1",
    "Footwear": "P1",

    "Golf": "P2",
    "Outdoors": "P2",
    "Fitness": "P2",

    "Fan Shop": "P3",
    "Technology": "P3",
    "Discs Shop": "P3",
    "Pet Shop": "P3",
    "Book Shop": "P3",
    "Health and Beauty": "P3"
}

In [65]:
product_plant_mapping = (
    dim_product[
        [
            "Product Card Id",
            "Product Name",
            "Department Name"
        ]
    ]
    .copy()
)

In [66]:
product_plant_mapping["Plant_ID"] = (
    product_plant_mapping["Department Name"]
    .map(department_to_plant)
)

In [67]:
print(product_plant_mapping.shape)

product_plant_mapping.head()

(118, 4)


,Product Card Id,Product Name,Department Name,Plant_ID
0,1360,Smart watch,Fitness,P2
48,365,Perfect Fitness Perfect Rip Deck,Apparel,P1
49,627,Under Armour Girls' Toddler Spine Surge Runni,Golf,P2
50,502,Nike Men's Dri-FIT Victory Golf Polo,Golf,P2
55,278,Under Armour Men's Compression EV SL Slide,Footwear,P1


In [68]:
product_plant_mapping["Plant_ID"].isna().sum()

np.int64(0)

In [69]:
product_plant_mapping["Plant_ID"].value_counts()

Plant_ID
P2    76
P1    23
P3    19
Name: count, dtype: int64

In [ ]:
'''
product_plant_mapping.to_csv(
    "../data/master/product_plant_mapping.csv",
    index=False
)
'''

In [70]:
plant_demand = (
    df.merge(
        product_plant_mapping[
            ["Product Card Id", "Plant_ID"]
        ],
        on="Product Card Id",
        how="left"
    )
    .groupby("Plant_ID")
    ["Order Item Quantity"]
    .sum()
    .sort_values(ascending=False)
)

plant_demand

Plant_ID
P1    141581
P2    131583
P3    110915
Name: Order Item Quantity, dtype: int64

Estimate Capacity

In [71]:
weekly_plant_demand = (
    df.merge(
        product_plant_mapping[
            ["Product Card Id", "Plant_ID"]
        ],
        on="Product Card Id",
        how="left"
    )
    .assign(
        Order_Date=pd.to_datetime(
            df["order date (DateOrders)"]
        )
    )
    .set_index("Order_Date")
    .groupby("Plant_ID")
    ["Order Item Quantity"]
    .resample("W")
    .sum()
    .reset_index()
)

In [72]:
weekly_plant_demand.groupby("Plant_ID")[
    "Order Item Quantity"
].describe()

,count,mean,std,min,25%,50%,75%,max
Plant_ID,,,,,,,,
P1,162.0,873.956790,274.604319,0.0,915.25,961.0,998.75,1151.0
P2,162.0,812.240741,282.901664,0.0,835.00,902.5,959.75,1048.0
P3,162.0,684.660494,152.530790,2.0,690.00,728.0,765.50,858.0


In [ ]:
'''
weekly_plant_demand.to_csv(
    "../data/processed/weekly_plant_demand.csv",
    index=False
)
'''

## Capacity Planning Assumption

The DataCo dataset does not contain manufacturing capacity information.

Weekly capacity was estimated using historical demand patterns and a target utilization rate of 75%.

Capacity = Average Weekly Demand / 0.75

This assumption creates realistic capacity constraints and enables bottleneck monitoring within the Control Tower.

In [74]:
capacity_table = (
    weekly_plant_demand
    .groupby("Plant_ID")["Order Item Quantity"]
    .mean()
    .reset_index()
)

capacity_table

,Plant_ID,Order Item Quantity
0,P1,873.956790
1,P2,812.240741
2,P3,684.660494


In [75]:
TARGET_UTILIZATION = 0.75

capacity_table["Weekly_Capacity"] = (
    capacity_table["Order Item Quantity"]
    / TARGET_UTILIZATION
).round().astype(int)

capacity_table

,Plant_ID,Order Item Quantity,Weekly_Capacity
0,P1,873.956790,1165
1,P2,812.240741,1083
2,P3,684.660494,913


In [76]:
capacity_table = capacity_table.rename(
    columns={
        "Order Item Quantity": "Avg_Weekly_Demand"
    }
)

capacity_table

,Plant_ID,Avg_Weekly_Demand,Weekly_Capacity
0,P1,873.956790,1165
1,P2,812.240741,1083
2,P3,684.660494,913


In [ ]:
'''
capacity_table.to_csv(
    "../data/master/capacity_table.csv",
    index=False
)
'''

In [78]:
pd.read_csv(
    "../data/master/capacity_table.csv"
)

,Plant_ID,Avg_Weekly_Demand,Weekly_Capacity
0,P1,873.956790,1165
1,P2,812.240741,1083
2,P3,684.660494,913
